# Retail shelf scoring: point-of-sale evaluation

This notebook analyzes photographs of retail shelves. It detects empty spots on the shelf and turns the detections, together with a photo-quality check, into a compliance score per image.

**Detections come from a pretrained Hugging Face YOLO model (no fine-tuning in this notebook).**

## Scores

| Score | Weight |
| --- | --- |
| Gap Score | 50% |
| Image Quality Score | 30% |
| Gap Density Score | 20% |

**Final Compliance Score** = weighted sum of the three.

## What the notebook does

1. Processes a full folder of images.
2. Runs inference with the pretrained YOLO model.
3. Calculates the weighted scores (Gap Score, Image Quality, Gap Density).
4. Handles problematic images gracefully.
5. Displays the results for each image and a summary for the whole folder.

In [1]:
# Import required libraries
#!pip install opencv-python ultralytics pandas huggingface_hub
import os
import glob
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

# The pretrained model has ONE class, named 'void' (an empty spot on the shelf).
# An earlier version of this notebook counted a class named 'gap', which the model does not
# have, so every image silently scored 100 for gaps. The check after loading the model
# now stops the run if the class name is ever wrong.
GAP_CLASS = 'void'

# Helper function to convert BGR (OpenCV) to RGB (for plotting)
def convert_bgr_to_rgb(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def calculate_gap_score(results, model):
    # Computing a simplified gap score based on the number of GAP_CLASS ('void') detections.
    # The score will be higher for fewer gaps.
    total_gaps = sum(1 for box in results.boxes if model.names[int(box.cls)] == GAP_CLASS)
    return max(0, 100 - (total_gaps * 10))

def calculate_gap_density_score(results, image_width, image_height, model):
    # Computing a simplified gap density score.
    # Higher density (more gaps relative to image size) means a lower score.
    gap_area = 0
    for box in results.boxes:
        if model.names[int(box.cls)] == GAP_CLASS:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            gap_area += (x2 - x1) * (y2 - y1)
            
    total_area = image_width * image_height
    if total_area == 0:
        return 0
    
    gap_density = gap_area / total_area
    return max(0, 100 - (gap_density * 100))

def calculate_image_quality_score(image):
    # Calculating image sharpness using the Laplacian variance method.
    # Returns a normalized score from 0 to 100, where higher is better.
    try:
        image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        laplacian_score = cv2.Laplacian(image_gray, cv2.CV_64F).var()
        
        # Normalizing score based on typical values. Adjust the 1000.0 as needed.
        normalized_score = min(100, (laplacian_score / 1000.0) * 100)
        return normalized_score
    except Exception as e:
        print(f"Error calculating image quality: {e}")
        return 0

def calculate_final_score(gap_score, quality_score, density_score):
    # Calculating the final weighted compliance score.
    return (0.50 * gap_score) + (0.30 * quality_score) + (0.20 * density_score)


# Downloading and loading the pretrained HUGGING FACE model
model_path = hf_hub_download(repo_id="akul-29/Retail-Shelf-Gap-Detection_Model", filename="best.pt")
model = YOLO(model_path)
print("Model classes:", model.names)
assert GAP_CLASS in model.names.values(), f"Model has no class '{GAP_CLASS}'. Available classes: {model.names}"

# Defining the folder containing images to be evaluated
# Original dataset dowloaded @ https://www.kaggle.com/datasets/humansintheloop/supermarket-shelves-dataset
image_folder = "shelve_images"
image_paths = glob.glob(os.path.join(image_folder, '*.jpg')) + glob.glob(os.path.join(image_folder, '*.png'))

if not image_paths:
    raise FileNotFoundError(f"No .jpg/.png images found in '{image_folder}'. Put the shelf images there (see README).")

# Creating a list to store the results for each image
all_results_data = []

# Processing all images and generating scores
for image_path in image_paths:
    #print(f"Processing {os.path.basename(image_path)}...")    # Uncomment if wish to see line per line progress
    
    image_cv2 = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if image_cv2 is None:
        print(f"Warning: Could not load image {image_path}. Skipping.")
        continue

    try:
        yolo_results = model(image_path, verbose=False)[0]
        h, w, _ = image_cv2.shape
        
        gap_score = calculate_gap_score(yolo_results, model)
        quality_score = calculate_image_quality_score(image_cv2)
        density_score = calculate_gap_density_score(yolo_results, w, h, model)
        
        final_score = calculate_final_score(gap_score, quality_score, density_score)
        
        all_results_data.append({
            'Image': os.path.basename(image_path),
            'Gap Score': round(gap_score, 2),
            'Image Quality Score': round(quality_score, 2),
            'Gap Density Score': round(density_score, 2),
            'Final Compliance Score': round(final_score, 2)
        })
    except Exception as e:
        print(f"Error processing {os.path.basename(image_path)}: {e}. Skipping.")

# Creating a Pandas DataFrame from the results
df = pd.DataFrame(all_results_data)

# Displaying the results for each photo
print("\n--- Scores for Each Photo ---")
pd.set_option('display.max_rows', None)
display(df)

# Computing and displaying the overall folder summary
if not df.empty:
    average_scores = df[['Gap Score', 'Image Quality Score', 'Gap Density Score', 'Final Compliance Score']].mean()
    print("\n--- Overall Folder Summary ---")
    display(average_scores.to_frame(name='Average Score'))
else:
    print("\nNo images were processed successfully.")


Model classes: {0: 'void'}



--- Scores for Each Photo ---


,Image,Gap Score,Image Quality Score,Gap Density Score,Final Compliance Score
0,001.jpg,60,51.29,95.169998,64.420000
1,002.jpg,100,81.73,100.000000,94.520000
2,003.jpg,100,16.92,100.000000,75.070000
3,004.jpg,40,100.00,96.820000,69.360001
4,005.jpg,70,82.87,97.699997,79.400000
5,006.jpg,80,12.03,94.980003,62.610000
6,007.jpg,100,52.89,100.000000,85.870000
7,008.jpg,0,100.00,96.639999,49.330002
8,009.jpg,90,100.00,99.389999,94.879997
9,010.jpg,90,47.06,99.779999,79.070000



--- Overall Folder Summary ---


,Average Score
Gap Score,78.000000
Image Quality Score,58.655556
Gap Density Score,98.732444
Final Compliance Score,76.343333
